# Norwegian fisheries seasonality — Gold consumer (A10)

**Question:** How do landed fish quantity and species composition vary by Norwegian
region and month?

This notebook consumes only the A9 Gold mart. The main view uses Norwegian landings
with actual landing year 2023. Quantity is **round weight in tonnes**: the source's
live-weight-equivalent kilograms divided by 1,000.

> **Preview status:** The results below use the complete local 2023 data, but the
> upstream Silver snapshot is not yet approved by Person B's quality process. This is
> ready for a local demonstration, not production publication.


## 1. Open the Gold candidate


In [ ]:
import json
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

ROOT = Path.cwd()
if not (ROOT / "src").is_dir():
    ROOT = ROOT.parent

spark = (
    SparkSession.builder
    .appName("a10-gold-consumer")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)

mart = ROOT / "data/preview/gold/fisheries_seasonality"
markers = list(mart.glob("model_version=*/input_version=*/_candidate.json"))
if len(markers) != 1:
    raise RuntimeError(f"Expected one Gold candidate, found {len(markers)}")

candidate = markers[0].parent
metadata = json.loads(markers[0].read_text(encoding="utf-8"))
if metadata["unvalidated_preview"]:
    print("WARNING: local unvalidated preview — not published Gold")

fact = spark.read.parquet(str(candidate / "fact_monthly_species_region"))
month = spark.read.parquet(str(candidate / "dim_month"))
region = spark.read.parquet(str(candidate / "dim_landing_region"))
species = spark.read.parquet(str(candidate / "dim_species"))

detail = fact.join(month, "month_id").join(region, "region_id").join(
    species, "species_id"
)
print(metadata["table_rows"])


## 2. Select the consumer view


In [ ]:
LANDING_YEAR = 2023
COUNTRY_CODE = "NOR"
COUNTY_CODE = None   # Example: "54" for Troms og Finnmark
SPECIES_CODE = None  # Example: "0201001" for cod

selected = detail.filter(
    (F.col("year") == LANDING_YEAR)
    & (F.col("landing_country_code") == COUNTRY_CODE)
)
if COUNTY_CODE:
    selected = selected.filter(F.col("landing_county_code") == COUNTY_CODE)
if SPECIES_CODE:
    selected = selected.filter(F.col("species_code") == SPECIES_CODE)


## 3. Coverage and headline result

| Metric | 2023 Norwegian landings |
|---|---:|
| Round weight | **2,492,144.338 tonnes** |
| Source product lines | 1,063,868 |
| Missing round weight | 1 line |
| Zero round weight | 168,031 lines |

Gold reconciles all **1,079,929** input Silver lines before the consumer filter.
Landing-year coverage is one line in 2022, 1,078,473 in 2023 and 1,455 in 2024.
Off-year landings are shown as coverage, not silently forced into 2023.


In [ ]:
headline = selected.agg(
    (F.sum("total_round_weight_kg") / 1000).alias("round_weight_tonnes"),
    F.sum("line_count").alias("line_count"),
    F.sum("missing_round_weight_rows").alias("missing_round_weight_rows"),
    F.sum("zero_round_weight_rows").alias("zero_round_weight_rows"),
)
headline.show(truncate=False)


## 4. Seasonal pattern

| Landing month | Round weight (tonnes) |
|---|---:|
| Jan | 232,513.779 |
| Feb | 226,964.179 |
| **Mar** | **377,637.044** |
| Apr | 231,849.480 |
| May | 148,521.423 |
| Jun | 183,910.202 |
| Jul | 108,794.002 |
| Aug | 203,165.021 |
| Sep | 185,572.568 |
| Oct | 282,289.153 |
| Nov | 241,040.534 |
| **Dec** | **69,886.952** |

March is the largest landing month by round weight; December is the smallest in the
selected view.


In [ ]:
monthly = (
    selected.groupBy("month_id")
    .agg(
        (F.sum("total_round_weight_kg") / 1000).alias("round_weight_tonnes"),
        F.sum("line_count").alias("line_count"),
    )
    .orderBy("month_id")
)
monthly.show(12, truncate=False)


## 5. Largest species by round weight

| Rank | Species | Round weight (tonnes) |
|---:|---|---:|
| 1 | Norwegian spring-spawning herring | **397,427.791** |
| 2 | Blue whiting | **383,698.045** |
| 3 | Cod | **344,357.115** |
| 4 | Mackerel | 342,080.095 |
| 5 | Saithe | 234,935.876 |

Cod has many more product lines than the two leading pelagic species, illustrating why
row count is not a quantity measure.


In [ ]:
top_species = (
    selected.groupBy("species_code", "species_name")
    .agg(
        (F.sum("total_round_weight_kg") / 1000).alias("round_weight_tonnes"),
        F.sum("line_count").alias("line_count"),
    )
    .orderBy(F.desc("round_weight_tonnes"))
)
top_species.show(10, truncate=False)


## 6. Largest landing counties by round weight

| Rank | County | Round weight (tonnes) |
|---:|---|---:|
| 1 | Troms og Finnmark | **607,963.588** |
| 2 | Rogaland | **573,913.652** |
| 3 | Møre og Romsdal | **507,485.081** |
| 4 | Nordland | 338,684.636 |
| 5 | Vestland | 283,055.291 |
| — | Unknown / not supplied | 30,064.795 |

Unknown county remains visible instead of being assigned to a region or removed.


In [ ]:
top_counties = (
    selected.groupBy("landing_county_code", "landing_county_name")
    .agg(
        (F.sum("total_round_weight_kg") / 1000).alias("round_weight_tonnes"),
        F.sum("line_count").alias("line_count"),
    )
    .orderBy(F.desc("round_weight_tonnes"))
)
top_counties.show(15, truncate=False)


## 7. Answer and limitations

The 2023 preview shows strong seasonality: landed round weight peaks in March, with
another high period in October–November. Quantity is concentrated in pelagic species,
especially Norwegian spring-spawning herring, blue whiting and mackerel, while cod is
the largest demersal species. Troms og Finnmark, Rogaland and Møre og Romsdal account
for the largest landed quantities.

Interpretation limits:

- A source row is a document product line, not one delivery or one fish.
- Round weight is the selected live-weight-equivalent measure; it must not be added to
  product or gross weight.
- Zero round weight is retained because its business meaning is not yet approved.
- Missing county and round weight remain explicit.
- These results must be regenerated after Person B's Silver quality rules, quarantine,
  validation and Gold publish gate are connected.
